In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import os
import mne
import torch
import scipy
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pympler import asizeof

from src import utils
from src.preprocessing import preprocessing

#list torch devices
for i in range(torch.cuda.device_count()):
    print(i+1,':',torch.cuda.get_device_name(i))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') ### cuda:NGPU
device


1 : NVIDIA RTX A6000
2 : NVIDIA RTX A6000
3 : NVIDIA RTX A6000
4 : NVIDIA RTX A6000


device(type='cuda')

In [10]:
from src.dataset import EEGDataset

data_folder= '/space/gzanardini/tuh'
preprocessed_dir = '/space/gzanardini/tuh'

h_data_folder = data_folder + '/Non-epileptic EDF/'
h_data_lookup_table = data_folder + '/info_h/TUH_Epilepsy_h_lookup_table.txt'
h_labels = data_folder + '/info_h/TUH_Epilepsy_h_text_label.csv'

pat_data_folder = data_folder + '/Epilepsy EDF/'
pat_data_lookup_table = data_folder + '/info_pat/TUH_Epilepsy_pat_lookup_table.txt'
pat_labels = data_folder + '/info_pat/TUH_Epilepsy_pat_text_label.csv'

window_length = 10 # seconds
overlap_factor = 1 # no overlap for the moment

h_dataset = EEGDataset(data_dir_path=h_data_folder, preprocessed_dir=preprocessed_dir, labels_path=h_labels, lookup_table_path=h_data_lookup_table, window_length=window_length, overlap_factor=overlap_factor)
pat_dataset = EEGDataset(data_dir_path=pat_data_folder, preprocessed_dir=preprocessed_dir, labels_path=pat_labels, lookup_table_path=pat_data_lookup_table, window_length=window_length, overlap_factor=overlap_factor)

do_preprocessing = False
if do_preprocessing:
    preprocessing(h_dataset)
    preprocessing(pat_dataset)
h_dataset.load_data()
pat_dataset.load_data()

pat_dataset.set_channel_name2channel_idx_dictionary(h_dataset.channel_name2channel_idx)
assert pat_dataset[0]['data'].shape == h_dataset[0]['data'].shape

print('nr healthy people: ', h_dataset.get_nr_patients())
print('nr patients:       ', pat_dataset.get_nr_patients())

print('nr healthy samples:  ', len(h_dataset))
print('nr patient samples:  ', len(pat_dataset))

print('size of h_dataset:   ', asizeof.asizeof(h_dataset)/(1024**3), 'GB')
print('size of pat_dataset: ', asizeof.asizeof(pat_dataset)/(1024**3), 'GB')



Loading EEG data: 100%|██████████| 1360/1360 [00:00<00:00, 4979.06it/s]


nr healthy people:  100
nr patients:        100
nr healthy samples:   25582
nr patient samples:   137050
size of h_dataset:    0.014892578125 GB
size of pat_dataset:  0.07695920765399933 GB


In [6]:
print(h_dataset.data_dir)
print(pat_dataset.data_dir)


/space/gzanardini/tuh/Non-epileptic EDF/
/space/gzanardini/tuh/Epilepsy EDF/
